# Part 6 — Interview & exam drill

⏱ **~1 hour, then come back to it before every interview**

**How to use this:**
1. **Whiteboard round:** write the code below from memory, with no peeking at `minitf/`.
2. **Question round:** answer each question **out loud**, as if to an interviewer, *before* opening the answer. A good answer takes 30–60 seconds and includes *why*, not just *what*.
3. Mark the ones you got wrong and redo them tomorrow.

Many questions link back to an experiment you ran. In an interview, *"I tested this: when I removed positional encoding, accuracy dropped from 100% to ~15%"* is much stronger than a textbook answer.

In [ ]:
# Setup
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "minitf").is_dir() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
import math
import torch
import torch.nn.functional as F
from minitf import check

## Whiteboard round

### ✍️ 1. Scaled dot-product attention with an optional mask, from memory (target: under 3 minutes)

In [ ]:
def attention_from_memory(q, k, v, mask=None):
    raise NotImplementedError("TODO: write it without looking")

check.check_sdpa(attention_from_memory)

### ✍️ 2. The causal mask, the shift for teacher forcing, and the parameter count of a Transformer block

In [ ]:
def causal_mask_from_memory(T):
    raise NotImplementedError("TODO: your code here")

def teacher_forcing_split(tgt):
    """tgt (B, T) = <bos> ... <eos>.  Return (decoder_input, targets)."""
    raise NotImplementedError("TODO: your code here")

def decoder_block_params(d, d_ff):
    """Masked self-attention + cross-attention + FFN + 3 LayerNorms (all with biases)."""
    raise NotImplementedError("TODO: your code here")

check.check_causal_mask(causal_mask_from_memory)
t = torch.tensor([[1, 5, 6, 7, 2]])
check.equal("teacher_forcing_split", [x.tolist() for x in teacher_forcing_split(t)], [[[1, 5, 6, 7]], [[5, 6, 7, 2]]])
from minitf._reference.transformer import DecoderBlock
check.equal("decoder block params (512, 2048)", decoder_block_params(512, 2048),
            sum(p.numel() for p in DecoderBlock(512, 8, 2048).parameters()))

### ✍️ 3. Explain on paper (no code)
- Draw the encoder-decoder Transformer with every shape.
- Give the shape of every tensor inside multi-head attention for B=2, T=10, d_model=512, h=8.
- Show why $\text{Var}(q\cdot k) = d_k$ when the components are independent with mean 0 and variance 1.

---
## Question round

### Tokens & embeddings

<details><summary><b>Q1.</b> What is an embedding? How is <code>nn.Embedding</code> related to a linear layer?</summary>

A learned dense vector for each token, i.e. a `(V, d)` matrix that you index into. Looking up row i equals `onehot(i) @ E`, so it's a bias-free linear layer applied to one-hot inputs, implemented as indexing for speed. Only the rows that are used get gradients.
</details>

<details><summary><b>Q2.</b> Why did word2vec-style embeddings work? Why does <code>king − man + woman ≈ queen</code>?</summary>

Distributional hypothesis: similar contexts give similar vectors. If context statistics decompose into attributes (gender, royalty...), the vectors become roughly linear in those attributes, so differences cancel the shared attributes and isolate one direction. (I checked this on a synthetic corpus with known attributes: the analogies came out exactly.)
</details>

<details><summary><b>Q3.</b> Static vs. contextual embeddings?</summary>

Static: one vector per word type ("bank" is always the same). Contextual: one vector per occurrence, computed from the surrounding tokens by attention layers.
</details>

<details><summary><b>Q4.</b> Why subword tokenization (BPE)? Characters or words would be simpler.</summary>

Word-level tokenization has a huge vocabulary and still has out-of-vocabulary words. Character-level gives very long sequences (and attention is O(T²)) with little meaning per token. BPE starts from characters and greedily merges frequent pairs, so common words become single tokens and rare words split into known pieces: an open vocabulary at a reasonable length. Byte-level BPE can represent any string.
</details>

<details><summary><b>Q5.</b> Why is word2vec trained with negative sampling?</summary>

A full softmax costs O(V) for every training pair with V ≈ 10⁶. Negative sampling turns it into binary classification against k random "negative" words, which costs O(k).
</details>

### Attention

<details><summary><b>Q6.</b> Explain self-attention as if to a smart non-expert.</summary>

Each word asks a question (query), and every word has a label (key) and content (value). A word compares its question with all the labels, turns the match scores into percentages with softmax, and takes that mix of the contents. So each word's new meaning is built from the words most relevant to it.
</details>

<details><summary><b>Q7.</b> Why divide by √d_k?</summary>

With unit-variance components the dot product has variance d_k. Large scores push the softmax into near one-hot saturation where gradients vanish. Dividing by √d_k restores variance 1 whatever the head size. (I measured the gradient collapsing by orders of magnitude without it.)
</details>

<details><summary><b>Q8.</b> Why separate Q, K, V projections? Why not use the embeddings directly?</summary>

With Q = K = V = X, every token mostly matches itself (x·x is largest), the pattern is symmetric, and there's no way to learn *what to look for* separately from *what to provide*. Separate projections decouple searching, matching and content.
</details>

<details><summary><b>Q9.</b> Why multi-head attention? What does it cost?</summary>

Several heads can attend to different relationships at once (one softmax = one weighted average). It costs nothing extra: d_model is split into h heads of d_model/h, so the parameters (4d² + 4d) and FLOPs are the same as a single head.
</details>

<details><summary><b>Q10.</b> Self- vs. cross-attention vs. masked self-attention?</summary>

Self: Q, K, V from the same sequence. Masked self: the same, plus a causal mask so position t can't see later positions (decoder). Cross: Q from the decoder, K and V from the encoder output, which is how the decoder reads the input.
</details>

<details><summary><b>Q11.</b> How is masking implemented, and why −∞?</summary>

Set the blocked scores to −∞ before the softmax, so exp gives 0 and the remaining weights still sum to 1. Zeroing after the softmax would break normalisation. A fully masked row gives NaN.
</details>

<details><summary><b>Q12.</b> What's the complexity of attention? How do people deal with long contexts?</summary>

O(T²·d) time and O(T²) memory per head per layer. FlashAttention computes it exactly, in tiles, without storing T×T (an IO optimisation; still O(T²) compute). Sparse, sliding-window and linear attention change the computation itself. Multi-query and grouped-query attention (MQA/GQA) shrink the KV cache.
</details>

<details><summary><b>Q13.</b> Is the attention matrix symmetric? Do rows or columns sum to 1?</summary>

Not symmetric (W_Q ≠ W_K). Rows sum to 1, since each query distributes one unit of attention.
</details>

<details><summary><b>Q14.</b> Why did attention replace RNNs?</summary>

(1) Parallel across positions during training (RNNs are sequential). (2) Any two tokens are one step apart (path length O(1) vs. O(T)), so long-range dependencies are easier and gradients don't have to pass through T steps. The cost: O(T²) in sequence length.
</details>

### Positions

<details><summary><b>Q15.</b> Why does a Transformer need positional encoding?</summary>

Self-attention is permutation-equivariant: shuffling the input just shuffles the output, so it can't tell "dog bites man" from "man bites dog". (In my ablation, removing it made `24/11/2026` and `2026/11/24` identical to the encoder, and accuracy collapsed from ~100% to ~15%.)
</details>

<details><summary><b>Q16.</b> Sinusoidal vs. learned vs. RoPE?</summary>

Sinusoidal: fixed sin/cos at geometric frequencies, added to the embeddings. No parameters, and PE(p+k) is a linear map of PE(p). Learned: a lookup table per position (BERT, GPT-2) that can't go beyond the trained length. RoPE: rotates q and k by position-dependent angles so q·k depends on the relative offset. It's the modern default (Llama and others).
</details>

<details><summary><b>Q17.</b> Can a decoder-only model work with no positional encoding at all?</summary>

To some extent, yes (NoPE): the causal mask leaks position, because position t attends over exactly t+1 tokens. An unmasked encoder gets no such signal.
</details>

### Architecture

<details><summary><b>Q18.</b> Walk through one encoder block and one decoder block.</summary>

Encoder: self-attention → Add & Norm → FFN → Add & Norm. Decoder: masked self-attention → Add & Norm → cross-attention (K, V from the encoder) → Add & Norm → FFN → Add & Norm. Finally a linear layer + softmax over the vocabulary.
</details>

<details><summary><b>Q19.</b> What does the FFN do? Why is it 4× wider?</summary>

A per-token MLP: it adds non-linearity and most of the capacity (≈⅔ of the block's parameters), and is often described as storing knowledge. The widening gives a larger hidden space to compute in. 4× is an empirical choice.
</details>

<details><summary><b>Q20.</b> Residual connections and LayerNorm: why both?</summary>

Residuals: a gradient highway, and each layer learns a small change (deep stacks become trainable). LayerNorm: normalises each token's vector to keep the activation scale stable.
</details>

<details><summary><b>Q21.</b> Pre-norm vs. post-norm?</summary>

Post-norm (paper) `LN(x + f(x))` can be unstable when deep and needs warm-up. Pre-norm `x + f(LN(x))` keeps an identity residual path, trains stably, and is used by GPT-2 onward and Llama (often with RMSNorm).
</details>

<details><summary><b>Q22.</b> Encoder-only vs. decoder-only vs. encoder-decoder?</summary>

BERT (bidirectional; classification, embeddings, retrieval). GPT / Llama / Claude (causal; generation, the dominant LLM design). T5 / original Transformer (seq2seq: translation, summarisation).
</details>

<details><summary><b>Q23.</b> Why multiply the embeddings by √d_model?</summary>

The embedding weights are small (shared with the output projection), so scaling brings token vectors to unit scale, comparable to the positional encodings.
</details>

<details><summary><b>Q24.</b> How many parameters does the base Transformer have, roughly?</summary>

~65M: d=512, h=8, N=6+6, d_ff=2048, 37k shared BPE vocabulary. Encoder block ≈ 3.15M, decoder block ≈ 4.2M, embeddings ≈ 19M.
</details>

### Training & inference

<details><summary><b>Q25.</b> What is teacher forcing? Why is training parallel?</summary>

Feed the ground truth shifted right as the decoder input and predict the sequence shifted left. All positions are computed in one pass, and the causal mask prevents peeking.
</details>

<details><summary><b>Q26.</b> What happens if you forget the causal mask?</summary>

Training loss becomes near 0 because the model copies the next token from its input, and inference is garbage. A suspiciously good training loss is a red flag.
</details>

<details><summary><b>Q27.</b> What should the loss be at initialisation?</summary>

≈ ln(V) (a uniform prediction). It's a free sanity check.
</details>

<details><summary><b>Q28.</b> Greedy, beam search, sampling (temperature, top-k, top-p)?</summary>

Greedy: argmax each step. Beam: keep the k best prefixes (translation). Sampling: draw from the softmax; temperature rescales the logits (lower is sharper); top-k / top-p restrict sampling to the most likely tokens.
</details>

<details><summary><b>Q29.</b> What is the KV cache?</summary>

During generation, cache each layer's K and V for past tokens. Each new step computes q, k, v only for the new token, so per-step compute is O(T) instead of re-running the whole prefix. Its memory (2 · layers · T · d per sequence) is a major serving bottleneck, which motivates GQA and MQA.
</details>

<details><summary><b>Q30.</b> What is exposure bias?</summary>

Training only ever sees correct prefixes, while inference sees the model's own (possibly wrong) outputs, so errors can compound.
</details>

<details><summary><b>Q31.</b> Label smoothing and warm-up in the paper?</summary>

Label smoothing ε = 0.1: the target puts 0.9 on the correct token and spreads the rest, which hurts perplexity but improves BLEU (less over-confidence). Warm-up: the learning rate rises linearly for 4000 steps, then decays as step^-0.5, which stabilises early training of post-norm Transformers with Adam.
</details>

### Curveballs

<details><summary><b>Q32.</b> If you double d_model, what happens to the parameters? If you double T, what happens to attention compute?</summary>

Parameters ≈ ×4 (they're ∝ d²). The attention-score compute is ≈ ×4 (∝ T²); the linear and FFN layers grow ×2 (∝ T).
</details>

<details><summary><b>Q33.</b> Can attention weights be read as "explanations"?</summary>

Only partly. Weights show where information is read from in that layer and head, but values are mixed again by W_O, residuals, FFNs and later layers. They're useful for spotting patterns (as in our cross-attention plots) but are not faithful attributions by themselves.
</details>

<details><summary><b>Q34.</b> Why might 1 head be enough for some tasks but not others?</summary>

If every output needs to read from essentially one place (our date task: copy the right digits), a single head can do it. Tasks that need several relationships at once (syntax + coreference + position) benefit from several heads.
</details>

<details><summary><b>Q35.</b> What does "contextual" mean in terms of your code?</summary>

After `src_embed`, the vector for "1" is the same wherever it appears. After even one encoder block, the vector at each "1" depends on the whole input through attention: the "1" in the day and the "1" in the month become different vectors.
</details>

---
**Done?** Go back to any notebook and change something: more heads, a pre-norm variant, learned positions, label smoothing. Predict the result, run it, and explain it. That loop of *predict, run, explain* is how you build intuition you can defend in an interview.